In [1]:
# ================================================================
# STEP 10.20 — STREAMLIT ANALYTICS BACKEND
# ================================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor

print("=" * 70)
print("STEP 10.20 — STREAMLIT ANALYTICS BACKEND")
print("=" * 70)

# ================================================================
# 1. LOAD DATASET
# ================================================================

data_path = r"C:\Users\hp\Desktop\nassau factory optimization\data\dataset.csv"

if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Dataset not found:\n{data_path}"
    )

df = pd.read_csv(data_path)

print("\nDataset loaded successfully.")
print("Shape:", df.shape)

# ================================================================
# 2. CREATE ANALYSIS DATAFRAME
# ================================================================

df_analysis = df.copy()

# Convert dates
df_analysis["Order Date"] = pd.to_datetime(
    df_analysis["Order Date"],
    dayfirst=True,
    errors="coerce"
)

df_analysis["Ship Date"] = pd.to_datetime(
    df_analysis["Ship Date"],
    dayfirst=True,
    errors="coerce"
)

# Shipping lead time
df_analysis["Shipping Lead Time"] = (
    df_analysis["Ship Date"]
    - df_analysis["Order Date"]
).dt.days

# Date features
df_analysis["Order Year"] = (
    df_analysis["Order Date"].dt.year
)

df_analysis["Order Month"] = (
    df_analysis["Order Date"].dt.month
)

df_analysis["Order Quarter"] = (
    df_analysis["Order Date"].dt.quarter
)

df_analysis["Order DayOfWeek"] = (
    df_analysis["Order Date"].dt.dayofweek
)

# ================================================================
# 3. PRODUCT → FACTORY MAPPING
# ================================================================

factory_mapping = {
    "Wonka Bar - Nutty Crunch Surprise": "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows": "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious": "Lot's O' Nuts",

    "Wonka Bar - Milk Chocolate": "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel": "Wicked Choccy's",

    "Laffy Taffy": "Sugar Shack",
    "SweeTARTS": "Sugar Shack",
    "Nerds": "Sugar Shack",
    "Fun Dip": "Sugar Shack",

    "Fizzy Lifting Drinks": "Sugar Shack",

    "Everlasting Gobstopper": "Secret Factory",

    "Hair Toffee": "The Other Factory",

    "Lickable Wallpaper": "Secret Factory",
    "Wonka Gum": "Secret Factory",

    "Kazookles": "The Other Factory"
}

df_analysis["Current Factory"] = (
    df_analysis["Product Name"]
    .map(factory_mapping)
)

# Check mapping
if df_analysis["Current Factory"].isna().any():
    missing_products = (
        df_analysis.loc[
            df_analysis["Current Factory"].isna(),
            "Product Name"
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        f"Unmapped products found: {missing_products}"
    )

print("\nFactory mapping completed.")
print(
    "Mapped records:",
    df_analysis["Current Factory"].notna().sum()
)

# ================================================================
# 4. ML FEATURES
# ================================================================

feature_columns = [
    "Product ID",
    "Division",
    "Current Factory",
    "Region",
    "Ship Mode",
    "Order Year",
    "Order Month",
    "Order Quarter",
    "Order DayOfWeek",
    "Units",
    "Sales",
    "Cost",
    "Gross Profit"
]

target_column = "Shipping Lead Time"

X = df_analysis[feature_columns].copy()
y = df_analysis[target_column].copy()

categorical_features = [
    "Product ID",
    "Division",
    "Current Factory",
    "Region",
    "Ship Mode"
]

numerical_features = [
    "Order Year",
    "Order Month",
    "Order Quarter",
    "Order DayOfWeek",
    "Units",
    "Sales",
    "Cost",
    "Gross Profit"
]

# ================================================================
# 5. TRAIN / TEST SPLIT
# ================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ================================================================
# 6. ENCODING
# ================================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# ================================================================
# 7. TRAIN FINAL GRADIENT BOOSTING MODEL
# ================================================================

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

gb_model.fit(
    X_train_encoded,
    y_train
)

print("\nGradient Boosting model trained successfully.")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Encoded features:", X_train_encoded.shape[1])

# ================================================================
# 8. DASHBOARD FILTER VALUES
# ================================================================

dashboard_products = sorted(
    df_analysis["Product Name"]
    .dropna()
    .unique()
    .tolist()
)

dashboard_regions = sorted(
    df_analysis["Region"]
    .dropna()
    .unique()
    .tolist()
)

dashboard_ship_modes = sorted(
    df_analysis["Ship Mode"]
    .dropna()
    .unique()
    .tolist()
)

dashboard_factories = sorted(
    df_analysis["Current Factory"]
    .dropna()
    .unique()
    .tolist()
)

print("\nDashboard filters prepared:")
print("Products:", len(dashboard_products))
print("Regions:", dashboard_regions)
print("Ship Modes:", dashboard_ship_modes)
print("Factories:", dashboard_factories)

# ================================================================
# 9. FACTORY COMPARISON FUNCTION
# ================================================================

def get_factory_comparison(
    product_name,
    region=None,
    ship_mode=None
):
    """
    Predict shipping lead time for a selected product
    under every available factory scenario.
    """

    filtered = df_analysis[
        df_analysis["Product Name"] == product_name
    ].copy()

    if region is not None and region != "All":
        filtered = filtered[
            filtered["Region"] == region
        ]

    if ship_mode is not None and ship_mode != "All":
        filtered = filtered[
            filtered["Ship Mode"] == ship_mode
        ]

    if filtered.empty:
        return pd.DataFrame()

    current_factory = (
        filtered["Current Factory"]
        .mode()
        .iloc[0]
    )

    actual_current_lead_time = (
        filtered["Shipping Lead Time"].mean()
    )

    results = []

    for factory in dashboard_factories:

        scenario = filtered.copy()

        # Hypothetical factory assignment
        scenario["Current Factory"] = factory

        X_scenario = scenario[
            feature_columns
        ].copy()

        X_scenario_encoded = (
            preprocessor.transform(X_scenario)
        )

        predicted_lead_time = (
            gb_model
            .predict(X_scenario_encoded)
            .mean()
        )

        results.append({
            "Product Name": product_name,
            "Factory": factory,
            "Current Factory": (
                "Yes"
                if factory == current_factory
                else "No"
            ),
            "Actual Current Lead Time":
                actual_current_lead_time,
            "Predicted Lead Time":
                predicted_lead_time,
            "Predicted Days Saved":
                actual_current_lead_time
                - predicted_lead_time
        })

    result = pd.DataFrame(results)

    numeric_columns = [
        "Actual Current Lead Time",
        "Predicted Lead Time",
        "Predicted Days Saved"
    ]

    result[numeric_columns] = (
        result[numeric_columns].round(2)
    )

    return result.sort_values(
        "Predicted Lead Time"
    ).reset_index(drop=True)

# ================================================================
# 10. PRODUCT RECOMMENDATION FUNCTION
# ================================================================

def get_product_recommendation(product_name):
    """
    Return recommendation information for a selected product.
    """

    if "final_recommendations" not in globals():
        return pd.DataFrame()

    return final_recommendations[
        final_recommendations["Product Name"]
        == product_name
    ].copy()

# ================================================================
# 11. REGIONAL PERFORMANCE FUNCTION
# ================================================================

def get_regional_performance(
    region=None,
    ship_mode=None
):
    """
    Return regional shipping performance.
    """

    filtered = df_analysis.copy()

    if region is not None and region != "All":
        filtered = filtered[
            filtered["Region"] == region
        ]

    if ship_mode is not None and ship_mode != "All":
        filtered = filtered[
            filtered["Ship Mode"] == ship_mode
        ]

    return (
        filtered
        .groupby("Region")
        .agg(
            Orders=("Order ID", "count"),
            Avg_Lead_Time=("Shipping Lead Time", "mean"),
            Avg_Sales=("Sales", "mean"),
            Avg_Gross_Profit=("Gross Profit", "mean"),
            Total_Units=("Units", "sum")
        )
        .round(2)
        .reset_index()
    )

# ================================================================
# 12. FACTORY PERFORMANCE FUNCTION
# ================================================================

def get_factory_performance():
    """
    Return observed factory performance.
    """

    return (
        df_analysis
        .groupby("Current Factory")
        .agg(
            Orders=("Order ID", "count"),
            Avg_Lead_Time=("Shipping Lead Time", "mean"),
            Median_Lead_Time=("Shipping Lead Time", "median"),
            Avg_Sales=("Sales", "mean"),
            Avg_Gross_Profit=("Gross Profit", "mean"),
            Total_Units=("Units", "sum")
        )
        .round(2)
        .reset_index()
        .sort_values("Avg_Lead_Time")
        .reset_index(drop=True)
    )

# ================================================================
# 13. BASIC KPI VALUES
# ================================================================

# These are calculated directly from the current final
# recommendation results.

dashboard_kpis = {
    "Total Records": len(df_analysis),
    "Products": df_analysis["Product Name"].nunique(),
    "Factories": df_analysis["Current Factory"].nunique(),
    "Average Lead Time": df_analysis["Shipping Lead Time"].mean()
}

print("\n" + "=" * 70)
print("DASHBOARD KPI VALUES")
print("=" * 70)

for key, value in dashboard_kpis.items():
    print(f"{key}: {value:.2f}")

# ================================================================
# 14. BACKEND TEST
# ================================================================

print("\n" + "=" * 70)
print("BACKEND FUNCTION TEST")
print("=" * 70)

test_product = dashboard_products[0]

print("Test product:", test_product)

test_result = get_factory_comparison(
    product_name=test_product
)

display(test_result)

print("\nStep 10.20 completed successfully.")

STEP 10.20 — STREAMLIT ANALYTICS BACKEND

Dataset loaded successfully.
Shape: (10194, 18)

Factory mapping completed.
Mapped records: 10194

Gradient Boosting model trained successfully.
Training rows: 8155
Testing rows: 2039
Encoded features: 39

Dashboard filters prepared:
Products: 15
Regions: ['Atlantic', 'Gulf', 'Interior', 'Pacific']
Ship Modes: ['First Class', 'Same Day', 'Second Class', 'Standard Class']
Factories: ["Lot's O' Nuts", 'Secret Factory', 'Sugar Shack', 'The Other Factory', "Wicked Choccy's"]

DASHBOARD KPI VALUES
Total Records: 10194.00
Products: 15.00
Factories: 5.00
Average Lead Time: 1320.84

BACKEND FUNCTION TEST
Test product: Everlasting Gobstopper


,Product Name,Factory,Current Factory,Actual Current Lead Time,Predicted Lead Time,Predicted Days Saved
0,Everlasting Gobstopper,Sugar Shack,No,1394.67,1397.22,-2.55
1,Everlasting Gobstopper,Secret Factory,Yes,1394.67,1414.00,-19.33
2,Everlasting Gobstopper,The Other Factory,No,1394.67,1414.00,-19.33
3,Everlasting Gobstopper,Wicked Choccy's,No,1394.67,1414.00,-19.33
4,Everlasting Gobstopper,Lot's O' Nuts,No,1394.67,1430.08,-35.41



Step 10.20 completed successfully.
